# RAG Pipeline – Application Integration

-  Retrieval-Augmented Generation (RAG) pipeline is a system that retrieves data from documents and uses it to answer questions
- Traditional Q&A systems search documents for keywords or matches and extracts exact text chunks and returns those text snippets.
- RAG Pipeline is a system that retrieves the relevant data from documents and then feeds the data to an LLM as context. The model reads the data and generates a conversational answer based on the source materials found.
-  RAG Pipeline consists of 3 steps:-
    1. Retrieval - system searches vector database to find relevant matches to the question
    2. Augmented - combines the relevant data found as context with the original query to send to the LLM
    3. Generation - LLM generates a conversational answer using the relevant data. 

##### _# Refer the question and answer, evaluation jupyter notebooks for more detailed notes._

#### _Build a Q&A System using Documents:_

Steps to Build Q&A System:-
1. Import nescessary tools from langchain
2. Prepare and Load application-relevant documents/data
3. Implement document retrieval using vector database
4. Pass retrieved context to the LLM using the QA Chain
5. Generate answers using retrieved information
6. Test retrieval quality

Import these -
- RetrivalQA from chain which will do retrival over documents
- Language model
- CVS loader from document loaders
- Doc array in memory search from vector stores which is an in memory vector store and does not need to connect to external database - there are many types of vector stores
- display and markdown

    from langchain.chains import RetrievalQA
    from langchain.chat_models import ChatOpenAI
    from langchain.document_loaders import CSVLoader
    from langchain.indexes import VectorstoreIndexCreator
    from langchain.vectorstores import DocArrayInMemorySearch


Load relevant documents:

    # Load Example csv file

    file = 'OutdoorClothingCatalog_1000.csv'
    loader = CSVLoader(file_path=file)
    data = loader.load()

Create an index - vector database for the document:

_Specify vector store class as docs array in memory search and take in list of documents to be loaded :_

    index = VectorstoreIndexCreator(
        vectorstore_cls=DocArrayInMemorySearch
    ).from_loaders([loader])

Specify the Q&A Chain with language model, chain type, retriver:

    llm = ChatOpenAI(temperature = 0.0, model=llm_model)
    qa = RetrievalQA.from_chain_type(
        llm=llm, 
        chain_type="stuff", 
        retriever=index.vectorstore.as_retriever(), 
        verbose=True,
        chain_type_kwargs = {
            "document_separator": "<<<<>>>>>"
        }
    )

#### _Example Queries:_

    examples = [
        {
            "query": "Do the Cozy Comfort Pullover Set\
            have side pockets?",
            "answer": "Yes"
        },
        {
            "query": "What collection is the Ultra-Lofty \
            850 Stretch Down Hooded Jacket from?",
            "answer": "The DownTek collection"
        }
    ]

#### _Test Q&A System with a query:_

    qa.run(examples[0]["query"])

#### _Manual evaluation with LangChain Debug:_

    import langchain
    langchain.debug = True
    qa.run(examples[0]["query"])
    # Turn off the debug mode
    langchain.debug = False

#### _LLM assisted Evaluation using QAEval Chain:_

=> QAEval Chain has built in system prompt to compare and evaluate answers.

Apply example queries with the QA Chain:

    predictions = qa.apply(examples)

Evaluate predicted LLM answers with real answers:

    from langchain.evaluation.qa import QAEvalChain

    llm = ChatOpenAI(temperature=0, model=llm_model)
    eval_chain = QAEvalChain.from_llm(llm)

    graded_outputs = eval_chain.evaluate(examples, predictions)

    print(graded_outputs)